# Задание 1. Журнал событий (лог-файл)

# Задание 1. Переработка программ лабораторной работы 4.1


In [10]:
from datetime import datetime
import os

LOG_FILE = "events.log"
REPORT_FILE = "report.txt"

def add_log(message):
    """Дописывает в журнал одну строку с текущей отметкой времени."""
    log_stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"{log_stamp} | {message}\n")

def show_log():
    """Выводит журнал на экран с нумерацией строк."""
    try:
        with open(LOG_FILE, "r", encoding="utf-8") as f:
            lines = f.readlines()
    except FileNotFoundError:
        print("Журнал пуст (файл не найден).")
        return

    if not lines:
        print("Журнал пуст.")
        return

    for i, line in enumerate(lines, start=1):
        print(f"{i}. {line.rstrip()}")

def count_log():
    """Возвращает количество записей в журнале."""
    try:
        with open(LOG_FILE, "r", encoding="utf-8") as f:
            return sum(1 for _ in f)
    except FileNotFoundError:
        return 0


def find_in_log(word):
    """Выводит только те записи, которые содержат заданное слово."""
    try:
        with open(LOG_FILE, "r", encoding="utf-8") as f:
            lines = f.readlines()
    except FileNotFoundError:
        print("Журнал пуст (файл не найден).")
        return

    found = [line.rstrip() for line in lines if word.lower() in line.lower()]
    if not found:
        print(f"Записей со словом «{word}» не найдено.")
        return

    for i, line in enumerate(found, start=1):
        print(f"{i}. {line}")


def file_stats(name):
    """
    Возвращает словарь со статистикой файла:
    строки, слова, символы, размер в байтах.
    """
    with open(name, "r", encoding="utf-8") as f:
        text = f.read()

    lines = text.splitlines()
    words = text.split()
    chars = len(text)
    size_bytes = os.path.getsize(name)

    return {
        "lines": len(lines),
        "words": len(words),
        "chars": chars,
        "bytes": size_bytes,
    }


def save_stats(stats, name):
    """Сохраняет статистику в файл отчёта (режим дозаписи)."""
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(name, "a", encoding="utf-8") as f:
        f.write(f"--- Отчёт от {stamp} ---\n")
        f.write(f"Строк:          {stats['lines']}\n")
        f.write(f"Слов:           {stats['words']}\n")
        f.write(f"Символов:       {stats['chars']}\n")
        f.write(f"Размер (байт):  {stats['bytes']}\n\n")


def file_stats_before(name):
    """Вариант 'до': с явным close()."""
    f = open(name, "r", encoding="utf-8")
    text = f.read()
    f.close()

    return {
        "lines": len(text.splitlines()),
        "words": len(text.split()),
        "chars": len(text),
        "bytes": os.path.getsize(name),
    }


def file_stats_after(name):
    """Вариант 'после': с оператором with."""
    with open(name, "r", encoding="utf-8") as f:
        text = f.read()

    return {
        "lines": len(text.splitlines()),
        "words": len(text.split()),
        "chars": len(text),
        "bytes": os.path.getsize(name),
    }


def demo_exception_closes_file():
    """
    Экспериментально показывает, что при исключении внутри with
    файл всё равно закрывается (f.closed == True).
    """
    print("=== Проверка авто-закрытия файла при исключении ===")
    f = None
    try:
        with open(LOG_FILE, "r", encoding="utf-8") as f:
            raise RuntimeError("Искусственное исключение внутри with")
    except RuntimeError as e:
        print(f"Поймано исключение: {e}")

    print(f"f.closed после выхода из with: {f.closed}")
    if f.closed:
        print("→ Файл закрыт автоматически, close() вручную не нужен.\n")


# ============================================================
#  ДЕМОНСТРАЦИЯ РАБОТЫ
# ============================================================

if __name__ == "__main__":
    # --- 1. Журнал событий ---
    add_log("Программа запущена")
    add_log("Файл открыт")
    add_log("Данные обработаны")
    add_log("Обнаружена ошибка")
    add_log("Программа завершена")

    print("=== Содержимое журнала ===")
    show_log()
    print(f"\nВсего записей: {count_log()}\n")

    print("=== Поиск по слову «ошибка» ===")
    find_in_log("ошибка")
    print()

    # --- 2. Статистика самого журнала ---
    stats = file_stats(LOG_FILE)
    print("=== Статистика файла events.log ===")
    print(f"Строк:          {stats['lines']}")
    print(f"Слов:           {stats['words']}")
    print(f"Символов:       {stats['chars']}")
    print(f"Размер (байт):  {stats['bytes']}\n")

    # Сохраняем статистику в отчёт
    save_stats(stats, REPORT_FILE)
    print(f"Статистика сохранена в {REPORT_FILE}\n")

    # --- 3. Сравнение "до" / "после" ---
    print("=== Сравнение вариантов file_stats ===")
    print(f"file_stats_before: {file_stats_before(LOG_FILE)}")
    print(f"file_stats_after:  {file_stats_after(LOG_FILE)}")
    print()

    # --- 4. Проверка авто-закрытия при исключении ---
    demo_exception_closes_file()

=== Содержимое журнала ===
1. 2026-09-19 13:41:45 | Программа запущена
2. 2026-09-19 13:41:45 | Файл открыт
3. 2026-09-19 13:41:45 | Данные обработаны
4. 2026-09-19 13:41:45 | Обнаружена ошибка
5. 2026-09-19 13:41:45 | Программа завершена

Всего записей: 5

=== Поиск по слову «ошибка» ===
1. 2026-09-19 13:41:45 | Обнаружена ошибка

=== Статистика файла events.log ===
Строк:          5
Слов:           25
Символов:       197
Размер (байт):  279

Статистика сохранена в report.txt

=== Сравнение вариантов file_stats ===
file_stats_before: {'lines': 5, 'words': 25, 'chars': 197, 'bytes': 279}
file_stats_after:  {'lines': 5, 'words': 25, 'chars': 197, 'bytes': 279}

=== Проверка авто-закрытия файла при исключении ===
Поймано исключение: Искусственное исключение внутри with
f.closed после выхода из with: True
→ Файл закрыт автоматически, close() вручную не нужен.



# Задание 2. Копирование файла с нумерацией строк

In [6]:
def copy_file(src, dst, skip_empty=False):
    """
    Копирует src в dst, добавляя номер строки в начало каждой строки.
    Возвращает (количество скопированных строк, количество пропущенных пустых строк).
    Если skip_empty=True — пустые строки пропускаются.
    """
    copied = 0
    skipped = 0

    with open(src, "r", encoding="utf-8") as fin, open(dst, "w", encoding="utf-8") as fout:

        for line in fin:
            clean = line.rstrip("\n").rstrip("\r")

            if skip_empty and clean.strip() == "":
                skipped += 1
                continue

            copied += 1
            fout.write(f"{copied}: {clean}\n")

    return copied, skipped


def show_file(name):
    """Выводит содержимое файла на экран."""
    print(f"=== Содержимое файла {name} ===")
    with open(name, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            print(f"{line.rstrip()}")
    print()


# ---------------- Демонстрация ----------------
if __name__ == "__main__":
    # Готовим исходный файл
    with open("source.txt", "w", encoding="utf-8") as f:
        f.write("Первая строка\n")
        f.write("Вторая строка\n")
        f.write("\n")                    # пустая строка
        f.write("Третья строка\n")
        f.write("\n")                    # ещё пустая
        f.write("Четвёртая строка\n")

    # Копирование с пропуском пустых строк
    n2, skipped = copy_file("source.txt", "dest1.txt", skip_empty=True)
    print(f"Скопировано строк: {n2}, пропущено пустых: {skipped}")
    show_file("dest1.txt")

Скопировано строк: 4, пропущено пустых: 2
=== Содержимое файла dest1.txt ===
1: Первая строка
2: Вторая строка
3: Третья строка
4: Четвёртая строка



# Задание 3. Поиск и замена текста в файле


In [7]:
import shutil


def replace_in_file(name, old, new):
    """
    Заменяет все вхождения old на new в файле name.
    Возвращает количество замен (0, если ничего не найдено).
    Перед изменением создаёт резервную копию name + '.bak'.
    """
    # 1) Читаем весь файл в память и закрываем его
    with open(name, "r", encoding="utf-8") as f:
        text = f.read()

    # 2) Считаем количество вхождений ДО замены
    count = text.count(old)

    # 3) Если нечего менять — выходим, ничего не трогаем
    if count == 0:
        print(f'Подстрока "{old}" не найдена, файл не изменён')
        return 0

    # 4) Делаем резервную копию (прочитать исходник → записать в .bak)
    backup = name + ".bak"
    shutil.copyfile(name, backup)
    print(f"Создана резервная копия: {backup}")

    # 5) Выполняем замену и перезаписываем файл
    new_text = text.replace(old, new)
    with open(name, "w", encoding="utf-8") as f:
        f.write(new_text)

    return count


def show_file(name):
    """Выводит содержимое файла на экран."""
    print(f"=== {name} ===")
    with open(name, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            print(f"{i}. {line.rstrip()}")
    print()


# ---------------- Демонстрация ----------------
if __name__ == "__main__":
    # Готовим исходный файл
    with open("text.txt", "w", encoding="utf-8") as f:
        f.write("Java — это язык программирования.\n")
        f.write("Java используется везде.\n")
        f.write("Многие любят Java.\n")
        f.write("Python — тоже хороший язык.\n")

    print(">>> Файл ДО изменений:")
    show_file("text.txt")

    # Замена №1 — «Java» → «Python»
    n = replace_in_file("text.txt", "Java", "Python")
    print(f"Выполнено замен: {n}\n")

    print(">>> Файл ПОСЛЕ первой замены:")
    show_file("text.txt")

    # Проверка: подстрока не найдена
    replace_in_file("text.txt", "Java", "C++")

>>> Файл ДО изменений:
=== text.txt ===
1. Java — это язык программирования.
2. Java используется везде.
3. Многие любят Java.
4. Python — тоже хороший язык.

Создана резервная копия: text.txt.bak
Выполнено замен: 3

>>> Файл ПОСЛЕ первой замены:
=== text.txt ===
1. Python — это язык программирования.
2. Python используется везде.
3. Многие любят Python.
4. Python — тоже хороший язык.

Подстрока "Java" не найдена, файл не изменён


# Задание 4. Мини-редактор текстового файла

In [9]:
import os

FILE = "notes.txt"


def ensure_file():
    """Создаёт пустой файл, если его ещё нет."""
    if not os.path.exists(FILE):
        with open(FILE, "w", encoding="utf-8"):
            pass  # просто создаём пустой файл


# ------------------------------------------------------------
#  Пункты меню
# ------------------------------------------------------------

def show_file():
    """Пункт 1 — показать содержимое с нумерацией строк."""
    ensure_file()
    with open(FILE, "r", encoding="utf-8") as f:
        lines = f.readlines()

    if not lines:
        print("Файл пуст.")
        return

    print(f"=== {FILE} ===")
    for i, line in enumerate(lines, 1):
        print(f"{i}. {line.rstrip()}")


def add_line():
    """Пункт 2 — добавить строку в конец файла (режим 'a')."""
    ensure_file()
    text = input("Введите текст строки: ")
    with open(FILE, "a", encoding="utf-8") as f:
        f.write(text + "\n")
    print("Строка добавлена.")


def clear_file():
    """Пункт 3 — очистить файл (режим 'w', сразу закрыть)."""
    with open(FILE, "w", encoding="utf-8"):
        pass
    print("Файл очищен.")


def show_stats():
    """Пункт 4 — статистика файла."""
    ensure_file()
    with open(FILE, "r", encoding="utf-8") as f:
        text = f.read()

    lines = text.splitlines()
    words = text.split()
    chars = len(text)
    size = os.path.getsize(FILE)

    print(f"Строк: {len(lines)}  Слов: {len(words)}  "
          f"Символов: {chars}  Размер: {size} байт")


def delete_line():
    """Пункт 5 — удалить строку по номеру."""
    ensure_file()
    with open(FILE, "r", encoding="utf-8") as f:
        lines = f.readlines()

    if not lines:
        print("Файл пуст — удалять нечего.")
        return

    # Показать текущее содержимое
    for i, line in enumerate(lines, 1):
        print(f"{i}. {line.rstrip()}")

    try:
        n = int(input("Введите номер строки для удаления: "))
    except ValueError:
        print("Ошибка: нужно ввести целое число.")
        return

    if n < 1 or n > len(lines):
        print(f"Ошибка: номер должен быть от 1 до {len(lines)}.")
        return

    del lines[n - 1]

    with open(FILE, "w", encoding="utf-8") as f:
        f.writelines(lines)

    print(f"Строка {n} удалена.")


# ------------------------------------------------------------
#  Меню
# ------------------------------------------------------------

def print_menu():
    print("\n===== МЕНЮ =====")
    print("1 - показать файл")
    print("2 - добавить строку")
    print("3 - очистить файл")
    print("4 - статистика")
    print("5 - удалить строку по номеру")
    print("0 - выход")


def main():
    while True:
        print_menu()
        choice = input("Ваш выбор: ").strip()

        if choice == "1":
            show_file()
        elif choice == "2":
            add_line()
        elif choice == "3":
            clear_file()
        elif choice == "4":
            show_stats()
        elif choice == "5":
            delete_line()
        elif choice == "0":
            print("Выход. До встречи!")
            break
        else:
            print("Некорректный ввод. Попробуйте снова.")


if __name__ == "__main__":
    main()


===== МЕНЮ =====
1 - показать файл
2 - добавить строку
3 - очистить файл
4 - статистика
5 - удалить строку по номеру
0 - выход


=== notes.txt ===
1. hello world
2. world hello

===== МЕНЮ =====
1 - показать файл
2 - добавить строку
3 - очистить файл
4 - статистика
5 - удалить строку по номеру
0 - выход
Строк: 2  Слов: 4  Символов: 24  Размер: 26 байт

===== МЕНЮ =====
1 - показать файл
2 - добавить строку
3 - очистить файл
4 - статистика
5 - удалить строку по номеру
0 - выход
1. hello world
2. world hello
Строка 2 удалена.

===== МЕНЮ =====
1 - показать файл
2 - добавить строку
3 - очистить файл
4 - статистика
5 - удалить строку по номеру
0 - выход
=== notes.txt ===
1. hello world

===== МЕНЮ =====
1 - показать файл
2 - добавить строку
3 - очистить файл
4 - статистика
5 - удалить строку по номеру
0 - выход
=== notes.txt ===
1. hello world

===== МЕНЮ =====
1 - показать файл
2 - добавить строку
3 - очистить файл
4 - статистика
5 - удалить строку по номеру
0 - выход
Выход. До встречи!
